[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/twistors/twistors.ipynb)

# Twistors and linked light

A twistor can describe a whole light ray. Events on that ray annihilate the twistor through their extensor action. This notebook builds that action, recovers a ray from its twistor, and uses one fixed twistor to select a ray through every event. Those rays carry a vacuum electromagnetic pulse whose electric and magnetic field lines are linked.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.relativity.twistors import render

In [ ]:
# Spacetime has three positive spatial directions and one negative time direction.
# A further positive and negative direction give its conformal algebra.
ga = Algebra("x+y+z+t-u+v-")
mv = NumpyContext(ga).multivector
exact = ga.exact.multivector
Scalar = ga.gatype.scalar()
Full = ga.gatype.full()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Event = ga.gatype.from_blades("x y z t")
Field = ga.gatype.from_blades("xy xz yz xt yt zt")
Twistor = ga.gatype.from_blades("1 y z u yz yu zu yzu")

## 1. Eight real components

Three commuting projectors select a space preserved by left multiplication, with eight independent coefficients per state. Embedding a state, multiplying it on the left and reading it back out gives every multivector's representation, with the multivector and the state left open; supplying the multivector gives its representation, a map from twistors to twistors. The conformal volume's representation squares to minus one and commutes with the representation of every even multivector.

In complex spinor notation the state has four complex components, and the conformal volume acts by multiplication by the imaginary unit.

In [ ]:
# Each factor halves the state space. The chosen blades keep its independent coefficients.
projector = (1 + exact.x) * (1 + exact.yt) * (1 + exact.zv) / 8
# Leaving Twistor open makes a linear map with one state input.
state_embedding = Twistor * projector                                       # [] Full <- Twistor
state_readout = (8 * Full).cast(Twistor)                                    # [] Twistor <- Full
representations = state_readout(Full * state_embedding)                      # [] Twistor <- (Full, Twistor)
# The volume's representation squares to minus the identity and commutes with every even one.
volume = representations(exact.xyztuv)                                      # [] Twistor <- Twistor

## 2. Events on one light ray share a twistor

An event acts on twistors by its representation, and the events on one light ray all send a shared twistor to zero. The twistor turned by the volume, `volume(twistor)`, belongs to the same ray: the two span a two-dimensional family of twistors for it.

In twistor notation that family is the twistor's nonzero complex multiples.

In [ ]:
emission = mv.x * -0.6 + mv.z * 0.3
light_direction = mv.t + (mv.x + mv.z) / np.sqrt(2)
separation = 1.4

origin = (mv.v - mv.u) / 2
infinity = mv.v + mv.u


def point(event: Event) -> Vector:
    return origin + event + event.squared() * infinity / 2


events = stack([emission, emission + light_direction * separation])         # [events] Event
points = point(events)                                                       # [events] Vector
# The product maps into both event kernels; its leading left singular states span their shared family.
family = representations(points[0] * points[1])                                     # [] Twistor <- Twistor
states, _, _ = family.svd()
twistor = states[0]                                                          # [] Twistor

## 3. Recover the ray and transform it

A pairing preserved by conformal rotors turns a twistor into its light-ray plane. Intersecting that plane with time slices recovers events. Transforming the twistor or the recovered ray gives the same geometry, following the [geometric-algebra construction of twistors](https://arxiv.org/abs/math-ph/0603037).

In twistor notation a twistor splits into two two-spinors, $Z = (\omega^A, \pi_{A'})$, and the events of its ray are the solutions $x$ of the incidence relation $\omega^A = i\,x^{AA'}\pi_{A'}$; the pairing is the Hermitian form $Z^\alpha \bar W_\alpha$ of signature $(2, 2)$, which vanishes on a null twistor.

In [ ]:
rapidity = 0.8
ray_samples = 120
margin = 0.6
fractions = np.linspace(-margin, 1 + margin, ray_samples)


def at_time(plane: Bivector, times: Scalar) -> Event:
    crossing = (mv.t - infinity * times) | plane
    return crossing.cast(Event) / -(infinity | crossing)


# A pairing of two twistors that conformal rotors preserve.
pairing = 8 * exact.yz.scalar_product(state_embedding.reverse() * exact.xyztuv * state_embedding)
# Pair the twistor with each plane's action, then raise that plane's input slot.
# Reverse the open plane because contract uses the reverse-product metric.
ray_readout = (Bivector * pairing(
    Twistor, volume(representations((1 * Bivector).reverse(), Twistor))
)).contract(1, 3)                                                            # [] Bivector <- (Twistor, Twistor)
plane = ray_readout(twistor, twistor)                                        # [] Bivector

event_times = -(mv.t | events)
ray_times = event_times[0] + (event_times[1] - event_times[0]) * fractions
ray = at_time(plane, ray_times)                                              # [ray_samples] Event
# Act on the twistor from one side, and on events by sandwiching with the rotor.
rotor = (mv.xt * (rapidity / 2)).exp()
moved_twistor = representations(rotor, twistor)
moved_plane = ray_readout(moved_twistor, moved_twistor)
moved_points = rotor >> points
moved_events = moved_points.cast(Event) / -(infinity | moved_points)
moved_event_times = -(mv.t | moved_events)
moved_ray_times = moved_event_times[0] + (moved_event_times[1] - moved_event_times[0]) * fractions
moved_ray = at_time(moved_plane, moved_ray_times)

render.draw_incidence(events, ray, moved_events, moved_ray);

## 4. One twistor chooses a ray through every event

A fixed non-null twistor selects a null twistor at each event by the event's action. Its ray passes through the event, and it pairs to zero with the fixed twistor and with that twistor turned by the volume. This family of straight light rays is the Robinson congruence. At one instant their spatial directions trace linked circles; these circles are instantaneous flow lines, not the trajectories of individual photons.

In twistor notation the Robinson congruence is the set of null twistors $Z$ orthogonal to a fixed non-null twistor $A$, $Z^\alpha \bar A_\alpha = 0$, Penrose's original construction.

In [ ]:
fixed_twistor = 1 + exact.y - exact.zu + exact.yzu
polars = np.pi * np.array([0.5, 0.65, 0.8])
per_circle = 10
fibre_samples = 240
azimuths = np.linspace(0, 2 * np.pi, per_circle, endpoint=False)
phases = np.linspace(-np.pi, np.pi, fibre_samples + 1)


def robinson(event: Event) -> Event:
    selected = representations(point(event), fixed_twistor)                          # [...] Twistor
    plane = ray_readout(selected, selected)
    tangent = (infinity | plane).cast(Event)
    return tangent / -(mv.t | tangent)


# Turn spinors along Hopf fibres, then stereographically project the circles into space.
starts = ((mv.zx * (-azimuths[None, :] / 2)).exp()
          * (mv.xy * (polars[:, None] / 2)).exp()).reshape((-1,))             # [curves] Even
orbit = starts[:, None] * (mv.xyz * mv.y * phases).exp()                     # [curves, fibre_samples + 1] Even
curves = -mv.xyz * orbit.select[2] / (1 + orbit.select[0])                   # [curves, fibre_samples + 1] Spatial

render.draw_congruence(curves);

## 5. A vacuum electromagnetic pulse

The same ray directions carry a null, source-free electromagnetic pulse with linked electric and magnetic field lines. Its energy travels along the twistor rays: at a probe point, the energy flow over the energy, plus the time direction, is the Robinson direction there. The magnetic lines are the electric family turned a quarter turn, and look alike; the electric lines are drawn. The field below follows [Swearngin and colleagues](https://arxiv.org/abs/1302.1431), with an overall normalization omitted.

In the notation of vector calculus the field is null, $\mathbf E \cdot \mathbf B = 0$ and $|\mathbf E| = |\mathbf B|$, and its energy flows along the Poynting vector $\mathbf E \times \mathbf B$.

In [ ]:
probe = mv.x * 0.4 + mv.z * 0.2
electric_turn = (mv.xy * (np.pi / 4)).exp()


def hopfion(event: Event) -> Field:
    position = -(event ^ mv.t) * mv.t
    time = -(event | mv.t)
    # The spinor turns the polarization and its propagation direction together.
    wave_spinor = 1 + mv.xyz * (position + time * mv.y)
    polarization = (mv.t + mv.y) ^ mv.x
    # The spacetime volume rotates electric into magnetic polarization. The inverse
    # supplies both the decaying amplitude and the local polarization angle.
    weight = (1 + event.squared() + 2 * mv.xyzt * (event | mv.t)).inverse()
    return (wave_spinor >> polarization) * weight.squared() * weight


field = hopfion(probe)                                                        # [] Field
electric = field | mv.t
magnetic = (mv.xyzt * field) | mv.t
energy = (electric.squared() + magnetic.squared()) / 2
energy_flow = -mv.xyz * (electric ^ magnetic)

# The energy's velocity at the probe, against the ray the fixed twistor selects there.
print(f"energy velocity:\n{(mv.t + energy_flow / energy).coefficient_table()}")
print(f"Robinson direction:\n{robinson(probe).coefficient_table()}")

# At time zero the electric lines are the flow lines, turned.
electric_curves = electric_turn >> curves
render.draw_fields(electric_curves);

## 6. Linked field lines in motion

The field lines are carried by the energy flow. Each material point travels along its straight light ray, while the whole linked curve deforms. Sampling those trajectories gives the pulse's changing field lines directly, without a time-stepping approximation.

In [ ]:
frames = 48
frame_duration = 80
frame_times = np.linspace(-1.25, 1.25, frames)

electric_rays = robinson(electric_curves)


def propagation() -> Iterator[Event]:
    for time in frame_times:
        yield electric_curves + electric_rays * time


render.inline(render.animate_fields(propagation()), frame_duration)

In [ ]:
# checks
# Both events send the twistor to zero, and the recovered ray meets them, before and after the boost.
np.testing.assert_allclose(representations(points, twistor).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose((at_time(plane, -(events | mv.t)) - events).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose((at_time(moved_plane, -(moved_events | mv.t)) - moved_events).kernel, 0.0, atol=1e-11)
# The pulse is null, and its energy flows along the Robinson ray at the probe.
np.testing.assert_allclose(field.squared().kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((mv.t + energy_flow / energy - robinson(probe)).kernel, 0.0, atol=1e-10)
